# Session 5 — TPU: Precision / Dtype Sweep (FP32 / BF16 / INT8)

## Background

Sessions 1–4 ran exclusively in FP32. Modern accelerators expose lower-precision math units that deliver higher FLOP/s at reduced memory footprint: NVIDIA Tensor Cores accelerate FP16 and BF16 matrix multiplications; TPU MXUs are native BF16. The tradeoff is numerical range and precision. FP16 has a narrow dynamic range and can underflow during training (addressed by loss scaling). BF16 shares FP32's exponent range, making it drop-in safe for most training workloads without a scaler.

**Note on FP16:** FP16 is not natively supported on TPU v5e (the MXU operates in BF16). This notebook tests FP32 and BF16 only for training, plus INT8 for inference.

**INT8 on TPU v5e:** The v5e MXU supports native INT8 operations (786 TOPS = 2× BF16 TFLOPS). Using `torch.autocast` with `dtype=torch.int8` on XLA or casting weights explicitly exercises this path.

## Goal

Measure how much throughput BF16 and INT8 gain over FP32 on the TPU v5litepod-1. The TPU MXU is native BF16, so a larger relative speedup is expected here than on the GPU for training.

## Question

How much throughput does BF16 add on the TPU relative to FP32? Does INT8 deliver the expected 2× over BF16?

---

**Runtime:** TPU (v5litepod-1 or equivalent)

After running, results are saved to `results/tpu_dtype_sweep.json` and loaded automatically by `03-analysis.ipynb`.

In [ ]:
import torch
import torch_xla.core.xla_model as xm

device = xm.xla_device()
print(f"Device  : {device}")
print(f"XLA device type: {xm.xla_device_hw(device)}")
print(f"PyTorch : {torch.__version__}")

import torch_xla
print(f"torch_xla: {torch_xla.__version__}")

In [ ]:
import time, torch.nn as nn
import torch_xla
import sys; sys.path.insert(0, "..")
from transformer_block import BenchmarkTransformerBlock, D_MODEL, N_HEAD, DIM_FEEDFORWARD

# ── Session 5 config ──────────────────────────────────────────────────────────
SEQ_LEN     = 128
BATCH_SIZES = [32, 64, 128, 256, 512, 1024]
DTYPES_TPU  = ["fp32", "bf16"]   # FP16 not natively supported on v5e
N_STEPS, WARMUP = 50, 5

DTYPE_MAP = {
    "fp32": torch.float32,
    "bf16": torch.bfloat16,
}

def benchmark(batch_size, dtype_str, device):
    dtype   = DTYPE_MAP[dtype_str]
    use_amp = dtype_str != "fp32"

    model     = BenchmarkTransformerBlock(
                    d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=DIM_FEEDFORWARD
                ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()
    model.train()

    elapsed = 0.0
    for step in range(N_STEPS + WARMUP):
        x = torch.randn(batch_size, SEQ_LEN, D_MODEL, device=device)
        y = torch.randn(batch_size, SEQ_LEN, D_MODEL, device=device)

        torch_xla.sync()
        t0 = time.perf_counter()

        optimizer.zero_grad()
        if use_amp:
            with torch.autocast("xla", dtype=dtype):
                loss = criterion(model(x), y)
        else:
            loss = criterion(model(x), y)

        loss.backward()
        optimizer.step()
        torch_xla.sync()

        t1 = time.perf_counter()
        if step >= WARMUP:
            elapsed += t1 - t0

    throughput = (N_STEPS * batch_size) / elapsed
    latency_ms = (elapsed / N_STEPS) * 1000
    return round(throughput, 1), round(latency_ms, 2)

print(f"Config  : seq_len={SEQ_LEN}, d_model={D_MODEL}")
print(f"Dtypes  : {DTYPES_TPU}")
print(f"Batches : {BATCH_SIZES}")
print("Benchmark function defined.")

In [ ]:
results = {d: {} for d in DTYPES_TPU}

for dtype_str in DTYPES_TPU:
    print(f"\n── {dtype_str.upper()} ──────────────────────────────────────")
    for bs in BATCH_SIZES:
        print(f"  batch={bs:<6} ... ", end="", flush=True)
        tput, lat = benchmark(bs, dtype_str, device)
        results[dtype_str][str(bs)] = {
            "throughput": tput,
            "latency_ms": lat,
        }
        print(f"{tput:>10,.0f} samples/sec  {lat:6.1f} ms")

print("\n── BF16 speedup vs FP32 ────────────────────────────")
for bs in BATCH_SIZES:
    fp32_val = results["fp32"].get(str(bs), {}).get("throughput")
    bf16_val = results["bf16"].get(str(bs), {}).get("throughput")
    if fp32_val and bf16_val:
        print(f"  batch={bs:<6}: {bf16_val / fp32_val:.3f}×")

---

## INT8 on TPU v5e  `# [PENDING RUN]`

The v5e MXU supports native INT8 arithmetic (786 TOPS = 2× the BF16 TFLOPS). To exercise this path with PyTorch/XLA, weights and activations must be cast to `torch.int8` before the XLA program is compiled. The simplest approach is to use `torch.autocast` with a custom INT8 policy, or to quantize Linear layers explicitly using `torch.ao.quantization` on the XLA device.

**Expected results (v5e):**
- Inference throughput: ~2× vs BF16 where the MXU is the bottleneck
- Training throughput: smaller gain (backward uses FP32/BF16 for gradient stability)

Run the cell below to benchmark. Results will be added to the saved JSON.

In [ ]:
# [PENDING RUN] INT8 on TPU v5e — inference throughput
# Cast model weights to int8 and run inference via PyTorch/XLA.
# XLA will route through the INT8 MXU path on v5e.

def benchmark_int8_tpu(batch_size, n_steps=N_STEPS, warmup=WARMUP):
    """Benchmark INT8 inference on TPU — forward pass only."""
    model_fp32 = BenchmarkTransformerBlock(
        d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=DIM_FEEDFORWARD
    )
    # Cast all linear layers to int8 before moving to XLA device
    # XLA will lower this to the v5e INT8 MXU path
    for module in model_fp32.modules():
        if isinstance(module, nn.Linear):
            module.weight.data = module.weight.data.to(torch.int8)
    model_int8 = model_fp32.to(device)
    model_int8.eval()

    elapsed = 0.0
    for step in range(n_steps + warmup):
        x = torch.randint(-128, 127, (batch_size, SEQ_LEN, D_MODEL),
                          dtype=torch.int8, device=device).float()
        torch_xla.sync()
        t0 = time.perf_counter()
        with torch.no_grad():
            _ = model_int8(x)
        torch_xla.sync()
        if step >= warmup:
            elapsed += time.perf_counter() - t0

    throughput = (n_steps * batch_size) / elapsed
    latency_ms = (elapsed / n_steps) * 1000
    return round(throughput, 1), round(latency_ms, 2)


print("── INT8 (v5e native) — Inference throughput ─────────")
results["int8"] = {}
for bs in BATCH_SIZES:
    print(f"  batch={bs:<6} ... ", end="", flush=True)
    try:
        tput, lat = benchmark_int8_tpu(bs)
        results["int8"][str(bs)] = {"throughput": tput, "latency_ms": lat, "mode": "inference"}
        print(f"{tput:>10,.0f} samples/sec  {lat:6.1f} ms")
    except Exception as e:
        results["int8"][str(bs)] = None
        print(f"ERROR: {e}")

print("\n── INT8 speedup vs FP32 (inference) ─────────────────")
for bs in BATCH_SIZES:
    fp32_r = results["fp32"].get(str(bs))
    int8_r = results["int8"].get(str(bs))
    if fp32_r and int8_r:
        ratio = int8_r["throughput"] / fp32_r["throughput"]
        print(f"  batch={bs:<6}: {ratio:.3f}×")

In [ ]:
import json, os
from datetime import datetime, timezone

os.makedirs("results", exist_ok=True)
payload = {
    "hardware":      "TPU",
    "device_name":   str(xm.xla_device_hw(device)),
    "session":       "5",
    "dtypes_tested": list(results.keys()),
    "batch_sizes":   BATCH_SIZES,
    "seq_len":       SEQ_LEN,
    "timestamp":     datetime.now(timezone.utc).isoformat(),
    "results":       results,
}
path = "results/tpu_dtype_sweep.json"
with open(path, "w") as f:
    json.dump(payload, f, indent=2)
print(f"Saved → {path}")